# Statistical Analysis — Pettitt Change-Point Test

This notebook applies the **Pettitt homogeneity test** to detect structural breaks
in benchmark metrics across ordered framework versions.

Unlike the Mann-Kendall trend test (which asks *"is there a monotonic trend?"*),
the Pettitt test asks *"is there a single change point — a version where the
distribution shifted?"*. Both tests are complementary.

**Test:** Pettitt (1979) — non-parametric, rank-based, single change-point.  
**p-value:** Monte Carlo simulation (`sim=20 000`).  
**Significance level:** alpha = 0.05.  
**Input file:** `summary_by_version.csv`.

**Reference:**  
Pettitt, A. N. (1979). A non-parametric approach to the change-point problem.
*Journal of the Royal Statistical Society: Series C*, 28(2), 126-135.
https://doi.org/10.2307/2346729


## 1. Setup

In [1]:
import subprocess, sys

for pkg in ["pyhomogeneity", "packaging", "plotly"]:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--quiet"], check=True)


In [2]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

# Local module — place pettitt_analysis.py in the same directory as this notebook
from pettitt_analysis import (
    load_summary,
    run_pettitt_test,
    plot_pettitt_heatmap,
    plot_pettitt_series,
    plot_pettitt_summary_bar,
    export_pettitt_results,
    METRICS,
    ALPHA,
)

print("pettitt_analysis loaded successfully")


pettitt_analysis loaded successfully


## 2. Load data

`load_summary` reads `summary_by_version.csv` and renames the `framework_version`
column to `version` automatically.


In [3]:
# Adjust the path if summary_by_version.csv is in a different location
CSV_PATH = "summary_by_version.csv"

df_summary = load_summary(CSV_PATH)

print(f"Shape          : {df_summary.shape}")
print(f"Frameworks     : {sorted(df_summary['framework'].unique())}")
print(f"Total versions : {df_summary['version'].nunique()}")
print(f"Columns (first 8): {df_summary.columns[:8].tolist()}")


Shape          : (573, 231)
Frameworks     : ['aiohttp', 'baize', 'django', 'fastapi', 'muffin', 'starlette']
Total versions : 539
Columns (first 8): ['framework', 'version', 'n_rounds', 'emission_duration_mean', 'emission_duration_std', 'emission_emissions_mean', 'emission_emissions_std', 'emission_cpu_energy_mean']


## 3. Run the Pettitt test

The test runs once per **(framework, metric)** combination. The p-value is
estimated via Monte Carlo simulation.

> **Tip:** set `sim=1000` for a quick preview, then `sim=20000` for the final
> publication run (takes a few minutes).


In [4]:
from pathlib import Path

PETTITT_CACHE = "pettitt_results.csv"  # path to the cache file
FORCE_RERUN   = True                  # set True to discard cache and rerun

if not FORCE_RERUN and Path(PETTITT_CACHE).exists():
    # ── Load from cache ──────────────────────────────────────────────────────
    pt_results = pd.read_csv(PETTITT_CACHE)
    pt_results["h"] = pt_results["h"].astype(bool)   # restore bool dtype
    print(f"Loaded from cache : {PETTITT_CACHE}")
else:
    # ── Run simulation ───────────────────────────────────────────────────────
    pt_results = run_pettitt_test(
        df_summary,
        metrics=METRICS,
        alpha=ALPHA,
        sim=20_000,      # reduce to 1_000 for quick testing
        min_versions=4,
    )
    # ── Save to cache ─────────────────────────────────────────────────────────
    pt_results.to_csv(PETTITT_CACHE, index=False)
    print(f"Simulation complete. Saved to: {PETTITT_CACHE}")

n_sig   = pt_results["h"].sum()
n_total = len(pt_results)
print(f"Tested   : {n_total} (framework, metric) pairs")
print(f"Detected : {n_sig} significant change points ({n_sig/n_total*100:.1f}%)")
pt_results.head(10)

Simulation complete. Saved to: pettitt_results.csv
Tested   : 78 (framework, metric) pairs
Detected : 71 significant change points (91.0%)


,framework,metric,n_versions,h,cp_index,cp_version,p_value,U_stat,mu1,mu2,delta,delta_pct
0,aiohttp,emission_energy_consumed_mean,61,True,37,3.11.14,0.0,836.0,0.000023,0.000022,-1.595411e-06,-6.846458
1,aiohttp,emission_cpu_energy_mean,61,True,37,3.11.14,0.0,852.0,0.000004,0.000004,-5.236623e-07,-11.923580
2,aiohttp,emission_ram_energy_mean,61,True,37,3.11.14,0.0,794.0,0.000019,0.000018,-1.071748e-06,-5.667359
3,aiohttp,emission_emissions_mean,61,True,37,3.11.14,0.0,836.0,0.000015,0.000014,-1.060865e-06,-6.846458
4,aiohttp,api_throughput_rps_mean,61,True,34,3.11.11,0.0,890.0,6329.191064,9351.158872,3.021968e+03,47.746509
5,aiohttp,api_rt_mean_mean,61,True,37,3.11.14,0.0,888.0,0.010493,0.006863,-3.629033e-03,-34.586830
6,aiohttp,api_rt_p95_mean,61,True,37,3.11.14,0.0,884.0,0.014245,0.010488,-3.756785e-03,-26.373132
7,aiohttp,html_throughput_rps_mean,61,True,28,3.11.5,0.0,858.0,8752.153797,10103.681613,1.351528e+03,15.442231
8,aiohttp,api_rt_mean_mean,61,True,37,3.11.14,0.0,888.0,0.010493,0.006863,-3.629033e-03,-34.586830
9,aiohttp,html_rt_p95_mean,61,True,28,3.11.5,0.0,853.0,0.008918,0.007614,-1.303813e-03,-14.619826


## 4. Heatmaps

Four views of the same results:
1. **Detection** — which pairs have a significant change point.
2. **p-value** — significance level.
3. **Relative change (delta%)** — direction and magnitude of the shift.
4. **Change-point version** — at which version the break occurs.


In [5]:
# 4a — Detection heatmap
fig = plot_pettitt_heatmap(pt_results, value="h")
fig.show(renderer="iframe_connected")


In [6]:
# 4b — p-value heatmap
fig = plot_pettitt_heatmap(pt_results, value="p_value")
fig.show(renderer="iframe_connected")


In [7]:
# 4c — Relative change at the break point
fig = plot_pettitt_heatmap(pt_results, value="delta_pct")
fig.show(renderer="iframe_connected")
fig.write_html("./plots/pettitt_heatmap_delta_pct.html")


In [8]:
# 4d — Which version is the change point
fig = plot_pettitt_heatmap(pt_results, value="cp_version")
fig.show(renderer="iframe_connected")
fig.write_html("./plots/pettitt_heatmap_cp_version.html")


## 5. Per-framework summary

In [9]:
fig = plot_pettitt_summary_bar(pt_results, metric_group="energy")
fig.show(renderer="iframe_connected")


In [10]:
fig = plot_pettitt_summary_bar(pt_results, metric_group="http")
fig.show(renderer="iframe_connected")


## 6. Drill-down: individual series with change-point annotation

Each plot shows the metric across versions (mean +/- std band) with:
- A **vertical dashed line** at the detected change point.
- Horizontal **mu1** (before) and **mu2** (after) reference lines.
- p-value and significance label in the annotation.


In [12]:
# Single framework / metric example
FRAMEWORK = "fastapi"
METRIC    = "emission_emissions_mean"

pt_row = pt_results[
    (pt_results["framework"] == FRAMEWORK) &
    (pt_results["metric"]    == METRIC)
]

fig = plot_pettitt_series(
    df_summary,
    framework=FRAMEWORK,
    metric=METRIC,
    pt_result_row=pt_row.iloc[0] if not pt_row.empty else None,
)
fig.show(renderer="iframe_connected")


In [13]:
# Loop: energy_consumed change-point series for all frameworks
for fw in sorted(df_summary["framework"].unique()):
    pt_row = pt_results[
        (pt_results["framework"] == fw) &
        (pt_results["metric"]    == "emission_emissions_mean")
    ]
    if pt_row.empty:
        continue
    fig = plot_pettitt_series(
        df_summary,
        framework=fw,
        metric="emission_emissions_mean",
        pt_result_row=pt_row.iloc[0],
    )
    #fig.write_html(f"./plots/{fw}/pettitt_emission_ecm_{fw}.html")
    fig.write_html(f"./plots/{fw}/pettitt_emission_emissions_{fw}.html")
print("Done.")

Done.


## 7. Significant results table

In [ ]:
sig = (
    pt_results[pt_results["h"] == True]
    .sort_values("p_value")
    [["framework", "metric", "cp_version", "p_value", "U_stat",
      "mu1", "mu2", "delta_pct", "n_versions"]]
    .reset_index(drop=True)
)

sig["p_value"]   = sig["p_value"].map(lambda x: "p < 0.0001" if x < 0.0001 else f"{x:.4f}")
sig["delta_pct"] = sig["delta_pct"].map(lambda x: f"{x:+.2f}%")
sig["mu1"]       = sig["mu1"].map(lambda x: f"{x:.4g}")
sig["mu2"]       = sig["mu2"].map(lambda x: f"{x:.4g}")

print(f"Significant change points: {len(sig)}")
display(sig)


Significant change points: 71


,framework,metric,cp_version,p_value,U_stat,mu1,mu2,delta_pct,n_versions
0,aiohttp,emission_energy_consumed_mean,3.11.14,0.0000,836.0,2.33e-05,2.171e-05,-6.85%,61
1,aiohttp,emission_cpu_energy_mean,3.11.14,0.0000,852.0,4.392e-06,3.868e-06,-11.92%,61
2,aiohttp,emission_ram_energy_mean,3.11.14,0.0000,794.0,1.891e-05,1.784e-05,-5.67%,61
3,aiohttp,emission_emissions_mean,3.11.14,0.0000,836.0,1.55e-05,1.443e-05,-6.85%,61
4,aiohttp,api_throughput_rps_mean,3.11.11,0.0000,890.0,6329,9351,+47.75%,61
...,...,...,...,...,...,...,...,...,...
66,muffin,emission_emissions_mean,0.98.5,0.0394,252.0,1.107e-05,1.124e-05,+1.54%,48
67,muffin,emission_energy_consumed_mean,0.98.5,0.0423,252.0,1.665e-05,1.691e-05,+1.54%,48
68,baize,api_rt_mean_mean,0.18.0,0.0440,120.0,0.007998,0.008456,+5.73%,30
69,baize,api_rt_mean_mean,0.18.0,0.0461,120.0,0.007998,0.008456,+5.73%,30


## 8. Export results

In [15]:
export_pettitt_results(
    pt_results,
    output_csv="pettitt_results.csv",
    output_html="./plots/pettitt_heatmap.html",
)
print("\nGenerated files:")
print("  pettitt_results.csv  — full results table")
print("  pettitt_heatmap.html — interactive heatmap (standalone HTML)")


Saved: pettitt_results.csv
Saved: ./plots/pettitt_heatmap.html

Generated files:
  pettitt_results.csv  — full results table
  pettitt_heatmap.html — interactive heatmap (standalone HTML)
